# Modelagem — Score de detração pós-viagem

Este notebook treina e avalia o modelo que estima, para cada jornada encerrada, a probabilidade de o Cliente se tornar Detrator. Ele começa onde o `pre-processamento.ipynb` termina: consome a base analítica já integrada e não repete nenhuma etapa de limpeza ou integração.

Princípios adotados:

- consumir a base analítica pronta, sem reprocessar as fontes brutas;
- importar a lógica de seleção, divisão e transformação de `scripts/preprocessamento_nps.py`, nunca reescrevê-la aqui;
- ajustar imputação, codificação e escala somente no treino;
- manter o conjunto de teste intocado até a etapa de avaliação;
- versionar o notebook sem saídas de execução e sem nenhum dado do parceiro.

O notebook é preenchido por etapas, cada uma com card próprio no board:

| Seção | Conteúdo | Card |
|---|---|---|
| 1 | Contrato de entrada e carga da matriz | #100 |
| 2 | Baselines de referência | #101 |
| 3 | Validação cruzada por Cliente | #102 |
| 4 | Primeiro modelo candidato | #103 |
| 5 | Ajuste de hiperparâmetros | #104 |
| 6 | Métricas de ordenação | #105 |
| 7 | Limiar operacional e matriz de confusão | #106 |
| 8 | Curvas ROC e precisão contra cobertura | #107 |
| 9 | Tabela comparativa | #108 |


## 1. Contrato de entrada

Esta seção fixa o que a etapa de carga e organização dos dados precisa entregar para que a modelagem rode. O contrato existe para que as duas frentes avancem em paralelo: enquanto a organização não estiver pronta, basta que o arquivo abaixo exista com estas colunas para que todo o restante do notebook funcione sem alteração.

**Arquivo esperado:** `data/processed/base_analitica.parquet`, produzido pela seção 13 do `notebooks/pre-processamento.ipynb`. No Colab, o mesmo arquivo pode vir do Drive; a busca de caminho está na célula seguinte e não exige edição de código.

**Colunas obrigatórias.** A base precisa carregar três grupos:

| Grupo | Colunas | Para que serve |
|---|---|---|
| Alvo | `DETRATOR` | variável a ser predita, binária |
| Chaves de partição | `DATA_STD_CONVERTIDA`, `ID_GOLDENRECORD` | corte temporal e agrupamento por Cliente |
| Features | as 11 de `FEATURE_SET_V1` | matriz do modelo |

As 11 features são a allowlist do score pós-viagem definida no módulo de pré-processamento e não são reescritas aqui: `TIER_VIAGEM`, `VOO_TIPO`, `TIPO_ENTRETENIMENTO`, `CANAL_COMPRA`, `SEGMENTO`, `ESTATISTICA_ATRASOSAIDA`, `ATRASO_CHEGADA`, `CANCELAMENTO_VOO`, `ANTECEDENCIA_CANCELAMENTO`, `TEMPO_VOO`, `N_TRECHOS`. A lista canônica é o `FEATURE_SET_V1`, que o módulo também expõe sob o nome antigo `FEATURES_SCORE_POS_VIAGEM`, e é ele que a célula 1.1 importa. O `QTDE_VIAGENS_12M` ocupava a última posição até o `feat(#149)` e está suspenso pela seção 4.3.2.3 da documentação, enquanto não houver forma auditável de recalcular o histórico com corte em `t_score`. Qualquer mudança nessa lista é decisão de modelagem e acontece no módulo, não no notebook.

**Partições.** A divisão não é aleatória. `split.dividir`, chamada dentro de `preparar_matriz` (seção 1.4), aplica corte temporal e por Cliente em três vias — treino, validação e teste —, retirando de cada conjunto anterior todo Cliente que reaparece num conjunto mais recente, porque a base tem dependência intracliente e um Cliente presente em dois lados inflaria o desempenho estimado. A função devolve as três partições como DataFrames completos, e é pelo índice de cada uma que features, alvo e grupos são referenciados no restante do notebook, nunca por posição.

**O que não pode entrar na matriz do modelo.** Nenhuma coluna iniciada por `NPS_` ou `SUB_`. São respostas coletadas no mesmo instrumento que origina o alvo, e o contrato temporal da seção 4.2.3 da documentação as exclui do modelo.

A proibição vale para a matriz, e não para a base analítica. A base carrega essas colunas por construção: `DETRATOR` é derivado de `NPS_PRINCIPAL` dentro da própria `preparar_base_analitica`, e a persistência em Parquet da seção 13 do pré-processamento não descarta coluna alguma. Exigir uma base sem elas faria o notebook falhar sobre uma entrada correta. Por isso as duas verificações têm papéis distintos, e isso é deliberado:

| Célula | Papel | Interrompe a execução |
|---|---|---|
| 1.3 | diagnóstico da base analítica: confere colunas obrigatórias e features da allowlist, e apenas relata quantas colunas de pesquisa vieram junto | sim para coluna obrigatória ausente, não para coluna de pesquisa presente |
| 1.4 | contrato da matriz do modelo: confere que o conjunto de features é exatamente a allowlist e que nenhuma coluna de pesquisa entrou, na entrada e na saída do pré-processador | sim |


### 1.1. Dependências e módulo de pré-processamento

In [ ]:
!pip install -q pandas numpy scikit-learn pyarrow matplotlib

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Inclui a pasta do módulo tanto no projeto local quanto no Colab. O break evita
# que duas cópias de preprocessamento_nps.py fiquem no caminho ao mesmo tempo,
# o que tornaria ambíguo qual delas o import resolve.
for pasta_codigo in [Path("scripts"), Path("../scripts"), Path("/content/scripts")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

import preprocessamento_nps
importlib.reload(preprocessamento_nps)

from preprocessamento_nps import (
    FEATURES_SCORE_POS_VIAGEM,
    validar_parquet,
)

SEMENTE = 42
np.random.seed(SEMENTE)

# O contrato da seção 1 ancora em 11 features. Uma mudança silenciosa na
# allowlist do módulo precisa falhar aqui, e não apenas aparecer no print.
QUANTIDADE_FEATURES_ESPERADA = 11
if len(FEATURES_SCORE_POS_VIAGEM) != QUANTIDADE_FEATURES_ESPERADA:
    raise AssertionError(
        f"A allowlist mudou: {len(FEATURES_SCORE_POS_VIAGEM)} features, "
        f"esperadas {QUANTIDADE_FEATURES_ESPERADA}. "
        "Atualize o contrato de entrada da seção 1 antes de seguir."
    )

print("features do score pós-viagem:", len(FEATURES_SCORE_POS_VIAGEM))


### 1.2. Localização e carga da base analítica

A busca cobre o projeto local e o Colab, com e sem Drive montado. Se nenhum caminho existir, a célula falha com a instrução do que fazer, em vez de seguir com uma base vazia.


In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
]

caminho_base = next((caminho for caminho in CAMINHOS_POSSIVEIS if caminho.exists()), None)

if caminho_base is None:
    raise FileNotFoundError(
        "Base analítica não encontrada. Gere-a com a seção 13 de notebooks/pre-processamento.ipynb "
        "ou monte o Drive e ajuste CAMINHOS_POSSIVEIS. Caminhos testados: "
        + ", ".join(str(caminho) for caminho in CAMINHOS_POSSIVEIS)
    )

validar_parquet(caminho_base)
df = pd.read_parquet(caminho_base)

print("base analítica:", caminho_base)
print("linhas:", f"{len(df):,}".replace(",", "."), "| colunas:", df.shape[1])


### 1.3. Diagnóstico da base analítica

A célula confere o que a base precisa ter e interrompe a execução se faltar coluna obrigatória ou feature da allowlist. Falhar aqui é barato; descobrir a ausência depois de treinar não é.

A contagem de colunas `NPS_` e `SUB_` é diagnóstico e não motivo de parada, porque a base analítica carrega essas colunas por construção, como explicado na seção 1. O que não pode acontecer é elas alcançarem a matriz do modelo, e é a seção 1.4 que bloqueia isso.


In [ ]:
COLUNAS_OBRIGATORIAS = {"DETRATOR", "DATA_STD_CONVERTIDA", "ID_GOLDENRECORD"}

faltando = COLUNAS_OBRIGATORIAS - set(df.columns)
if faltando:
    raise KeyError(f"A base não atende ao contrato de entrada. Colunas ausentes: {sorted(faltando)}")

features_ausentes = [coluna for coluna in FEATURES_SCORE_POS_VIAGEM if coluna not in df.columns]
if features_ausentes:
    raise KeyError(f"Features da allowlist ausentes na base: {features_ausentes}")

# Diagnóstico, e não trava: a base analítica carrega as respostas da pesquisa
# por construção, já que DETRATOR é derivado de NPS_PRINCIPAL. O bloqueio
# efetivo acontece sobre a matriz do modelo, na seção 1.4.
colunas_pesquisa = [coluna for coluna in df.columns if coluna.startswith(("NPS_", "SUB_"))]
print("colunas de pesquisa na base analítica:", len(colunas_pesquisa), "(diagnóstico, não bloqueia)")
print("prevalência de Detrator:", f"{df['DETRATOR'].mean():.2%}")


### 1.4. Matriz de modelagem e contrato da matriz

`preparar_matriz`, de `src/matriz.py`, executa numa chamada as três etapas que precisam
acontecer nesta ordem: particionar por data e por Cliente, selecionar a allowlist de features e
ajustar o pré-processador **apenas no treino**. Inverter as duas últimas seria inofensivo;
ajustar o pré-processador antes de particionar não, porque a mediana do treino passaria a
carregar informação de 2026 e a métrica de teste mediria um modelo que já viu o período que
deveria prever.

As datas de corte não são escolhidas aqui. Elas vêm do registro de decisão em
`documents/extras/politica-de-particionamento-temporal.md`, que é a fonte única: repeti-las como
constante do notebook criaria uma segunda verdade que poderia divergir da documentada.

`sem_data="treino"` envia ao treino qualquer linha sem data — hoje cerca de 120 mil na base
analítica, remanescentes de respostas antigas que a Azul não datou. Enviá-las ao treino em vez
de descartá-las evita perder um quinto da base; a garantia de que isso não corrompe o corte
temporal vem de `verificar_anterioridade_sem_data`, que prova pela ordem de registro
(`RESPONDENT_ID`) que essas linhas antecedem o período datado. `split.dividir` chama essa
verificação por conta própria sempre que existir alguma linha sem data, interrompendo a
execução se a prova deixar de valer.

O contrato da matriz é cobrado nas duas pontas do pré-processador. Na entrada, pega uma allowlist
alterada; na saída, uma transformação que reintroduza uma coluna de pesquisa sob outro nome.
Qualquer uma das duas interrompe a execução.


In [ ]:
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

import matriz
importlib.reload(matriz)
from matriz import preparar_matriz, resumo_da_matriz

# Datas do registro de decisão do #127, não escolhidas aqui.
CORTE_VALIDACAO = "2025-07-01"
CORTE_TESTE = "2026-01-01"

preparo = preparar_matriz(
    df,
    corte_validacao=CORTE_VALIDACAO,
    corte_teste=CORTE_TESTE,
    sem_data="treino",
)

x_treino, x_validacao, x_teste = (preparo["x"][k] for k in ("treino", "validacao", "teste"))
y_treino, y_validacao, y_teste = (preparo["y"][k] for k in ("treino", "validacao", "teste"))
grupos_treino = preparo["grupos"]["treino"]
preprocessador = preparo["preprocessador"]
x_treino_transformado = preparo["matrizes"]["treino"]
x_validacao_transformado = preparo["matrizes"]["validacao"]
x_teste_transformado = preparo["matrizes"]["teste"]
numericas = preparo["metadados"]["numericas"]
categoricas = preparo["metadados"]["categoricas"]
metadados_divisao = preparo["metadados"]

display(resumo_da_matriz(preparo))

anterioridade = metadados_divisao["anterioridade_sem_data"]
# verificar_anterioridade_sem_data devolve dois formatos: um quando ha linha
# sem data para verificar, outro (sem as chaves de correlacao) quando o bloco
# esta vazio, que e o caso normal na base atual.
if anterioridade["linhas_sem_data"]:
    print("linhas sem data enviadas ao treino:", f"{anterioridade['linhas_sem_data']:,}".replace(",", "."))
    print("  correlação de Spearman entre ordem de registro e data:", anterioridade["correlacao_spearman"])
    print("  maior id sem data:", anterioridade["maior_id_sem_data"],
          "< menor id datado:", anterioridade["menor_id_datado"])
else:
    print("nenhuma linha sem data na base: a verificação de anterioridade não se aplica")
print("removidas por Cliente recorrente:",
      f"{metadados_divisao['linhas_removidas_por_recorrencia']:,}".replace(",", "."))
print("features:", len(metadados_divisao["features_usadas"]),
      "| numéricas:", len(numericas), "| categóricas:", len(categoricas),
      "| colunas da matriz:", metadados_divisao["colunas_da_matriz"])


### 1.5. Composição das partições (CR05 do #127/#128)

Esta seção não reparticiona: `split.resumo` recebe `preparo["particoes"]` e
`preparo["metadados"]`, já calculados na 1.4, e devolve n, intervalo de datas,
Clientes e percentual do total por partição. Chamar `split.dividir` de novo aqui
geraria a mesma divisão duas vezes no mesmo notebook, com o risco de as duas
chamadas divergirem se uma delas for editada e a outra não.


In [ ]:
from split import resumo

tabela_particoes = resumo(preparo["particoes"], metadados=preparo["metadados"])
print(tabela_particoes)


## 2. Baselines de referência

Um número de desempenho só significa alguma coisa quando existe um piso para compará-lo. Esta seção estabelece dois pisos, e todo modelo apresentado depois é julgado contra eles.

O primeiro é a **regra da classe majoritária**, que não aprende nada e existe para expor o quanto a acurácia engana numa base em que cerca de um em cada cinco Clientes é Detrator. O segundo é uma **regressão logística**, modelo linear e interpretável, que mede quanto do sinal já é capturado sem recorrer a interação entre variáveis. É contra esse segundo piso que a escolha de um método baseado em árvores, na seção 4, precisa se justificar.

Os dois são ajustados apenas na partição de treino e medidos sobre a mesma partição de teste que o modelo candidato usará, sem o que a comparação não seria válida.

**Escopo desta seção.** A acurácia e a precisão média aparecem aqui como piso de comparação, não como a resposta ao item (c) da Seção 4.3. A escolha e a explicação das três métricas oficiais do modelo são daquele item e ficam na seção 6 deste notebook.

### 2.1. Prevalência e função de avaliação

A prevalência da classe Detrator é o número que dá sentido a todo o resto: é ela que a precisão média de um modelo sem informação reproduz, e é dela que a acurácia do baseline trivial se aproxima por complemento.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, average_precision_score,
                             roc_auc_score)

print(f"prevalência de Detrator no treino: {y_treino.mean():.2%}")
print(f"prevalência de Detrator no teste:  {y_teste.mean():.2%}")


def avaliar_no_teste(nome, modelo):
    """Mede um modelo já treinado sobre a partição de teste temporal.

    A acurácia entra ao lado da precisão média de propósito. É a distância
    entre as duas que evidencia o efeito do desbalanceamento, e é isso que
    justifica não apresentar acurácia como medida de qualidade neste problema.

    O ROC-AUC entra porque a tabela comparativa da seção 9 cobra as três
    métricas oficiais definidas na seção 4.1.3, e ele é a única das três que
    já pode ser medida aqui: depende apenas da ordenação do score. Revocação
    em Detrator e precisão dependem do ponto de corte, que só é decidido na
    seção 7, e medi-las agora seria fixar um limiar arbitrário antes da
    seção que existe para escolhê-lo.
    """
    probabilidades = modelo.predict_proba(x_teste_transformado)[:, 1]
    return {
        "modelo": nome,
        "acuracia": accuracy_score(y_teste, modelo.predict(x_teste_transformado)),
        "precisao_media": average_precision_score(y_teste, probabilidades),
        "roc_auc": roc_auc_score(y_teste, probabilidades),
    }

### 2.2. Baseline trivial: regra da classe majoritária

O classificador responde sempre a classe mais frequente do treino, que é a de não Detrator. Ele não olha nenhuma feature.

O resultado esperado é uma acurácia próxima de quatro quintos, acompanhada de uma precisão média próxima da prevalência. É essa contradição que interessa: um modelo que erra todos os Detratores da base ainda assim exibe acurácia alta. A probabilidade que ele emite é degenerada, constante para toda linha, e é por isso que a precisão média desaba para o piso. Pela mesma razão o ROC-AUC fica em 0,5: sem variação no score não existe ordenação, e 0,5 é exatamente o valor de quem ordena ao acaso.

In [ ]:
trivial = DummyClassifier(strategy="most_frequent", random_state=SEMENTE)
trivial.fit(x_treino_transformado, y_treino)

resultado_trivial = avaliar_no_teste("classe majoritária", trivial)
print(f"acurácia:       {resultado_trivial['acuracia']:.4f}")
print(f"precisão média: {resultado_trivial['precisao_media']:.4f}")
print(f"ROC-AUC:        {resultado_trivial['roc_auc']:.4f}")

### 2.3. Baseline linear: regressão logística

O segundo piso já aprende, mas apenas efeitos aditivos: cada variável contribui com um peso fixo, sem que o modelo represente combinações entre elas.

O parâmetro de balanceamento de classe é declarado explicitamente. Sem ele a logística se ajusta à classe majoritária e passa a quase não prever Detrator, o que a tornaria indistinguível do baseline trivial e inútil como piso. A matriz chega aqui já escalonada pelo pré-processador da seção 1.4, condição para que os coeficientes sejam comparáveis entre variáveis de unidades diferentes.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistica = LogisticRegression(
    class_weight="balanced",   # declarado: sem isso o modelo colapsa na classe majoritária
    # O padrão de max_iter é 1000 e não basta aqui: o lbfgs ainda não
    # convergiu nesse limite, e um modelo interrompido antes da convergência
    # não serve como piso, porque os coeficientes ainda estariam mudando.
    # 5000 dá folga sem custo relevante nesta matriz.
    max_iter=5000,
    random_state=SEMENTE,
)
logistica.fit(x_treino_transformado, y_treino)

resultado_logistica = avaliar_no_teste("regressão logística", logistica)
print(f"acurácia:       {resultado_logistica['acuracia']:.4f}")
print(f"precisão média: {resultado_logistica['precisao_media']:.4f}")
print(f"ROC-AUC:        {resultado_logistica['roc_auc']:.4f}")

### 2.4. Coeficientes da regressão logística

Os coeficientes ficam acessíveis no notebook para leitura posterior. Eles indicam a direção com que cada variável empurra o risco de detração, e voltam a ser usados na discussão de resultados, quando for preciso dizer se o modelo candidato encontrou algo que o piso linear não via.

In [ ]:
coeficientes_logistica = pd.DataFrame({
    "feature": preprocessador.get_feature_names_out(),
    "coeficiente": logistica.coef_[0],
}).sort_values("coeficiente", key=abs, ascending=False, ignore_index=True)

print("intercepto:", float(logistica.intercept_[0]))
coeficientes_logistica.head(15)

### 2.5. Resultado consolidado dos dois pisos

A tabela abaixo é o piso de comparação que as seções seguintes consomem. A leitura que importa: o baseline trivial exibe a maior acurácia da tabela e a menor precisão média, e a regressão logística inverte essa relação. O ROC-AUC completa a leitura: o trivial fica no 0,5 de quem não ordena nada, e é a distância da logística para esse 0,5 que mede o sinal já capturado por um modelo puramente aditivo. Qualquer modelo apresentado depois precisa superar a segunda linha na precisão média para que seu ganho seja atribuível ao modelo, e não ao desbalanceamento da base.

In [ ]:
tabela_baselines = pd.DataFrame([resultado_trivial, resultado_logistica])

ganho = resultado_logistica["precisao_media"] - resultado_trivial["precisao_media"]
print(f"ganho da logística sobre o trivial na precisão média: {ganho:+.4f}")
if ganho <= 0:
    print("ATENÇÃO: a logística não superou o piso trivial; revise o balanceamento antes de seguir.")

tabela_baselines

## 3. Validação cruzada por Cliente

`GroupKFold` de 5 folds dentro do treino, agrupando por `ID_GOLDENRECORD`.

**Por que agrupada, e não aleatória.** As respostas de um mesmo Cliente não são independentes: a
hipótese 4 da seção 4.2.3 mediu que quem detratou uma vez volta a detratar com chance 4,75 vezes
maior, e que o efeito sobrevive ao controle por atraso e cancelamento. Num `KFold` comum, duas
respostas da mesma pessoa cairiam em folds diferentes, e o modelo acertaria a segunda por
reconhecer a primeira. A média dos cinco folds subiria sem que nada tivesse melhorado, e o
hiperparâmetro escolhido em cima dela seria escolhido sobre um número inflado.

É a mesma razão que governa a divisão treino/teste em `split.dividir`, aplicada um nível abaixo.
Aqui ela é ainda mais necessária: o corte temporal não protege nada dentro do treino, porque os
folds não são temporais.

**Por que folds e não um terceiro bloco temporal.** A partição já reserva os meses finais para o
teste; recortar mais um bloco encolheria o treino justamente na faixa em que o histórico de
Cliente começa a existir. Com os folds, cada linha do treino serve de validação exatamente uma
vez, sem sair do ajuste.

A conferência abaixo não é decorativa: `conferir_folds` interrompe a execução se um Cliente
aparecer na validação de dois folds, ou no ajuste e na validação do mesmo fold. O `GroupKFold`
garante as duas coisas por construção, mas a garantia depende de o vetor de grupos ser o certo, e
passar o vetor errado produz folds tecnicamente válidos e cientificamente errados.

In [ ]:
from validacao import (conferir_folds, criar_folds, metadados_da_divisao,
                       resumo_dos_folds)

# Os indices dos folds sao posicionais, entao valem igual para x_treino e para
# x_treino_transformado: preparar_matriz constroi os dois a partir das mesmas
# linhas, na mesma ordem.
folds = criar_folds(x_treino, grupos_treino)
conferir_folds(folds, grupos_treino)

print(resumo_dos_folds(folds, grupos_treino, y_treino))
print()
for chave, valor in metadados_da_divisao(preparo).items():
    print(f"{chave}: {valor}")

## 4. Primeiro modelo candidato

Classificador de gradient boosting sobre árvores, escolha justificada pela ausência de preditor dominante e pela interação entre fidelização e falha operacional documentadas na seção 4.2.1.

*A preencher no card #103.*


### 4.1. Hiperparâmetros declarados e treino do candidato

`HIPERPARAMETROS_CANDIDATO` declara todos os hiperparâmetros que importam, inclusive os que
coincidem com o padrão da biblioteca, porque padrão não declarado é decisão que ninguém tomou.
Dois deles merecem leitura antes de rodar:

- `early_stopping=False`. No padrão `"auto"` o `HistGradientBoostingClassifier` liga a parada
  antecipada sozinho acima de dez mil linhas e separa uma fatia **aleatória** do ajuste para
  medir. Essa fatia não respeita `ID_GOLDENRECORD`, então respostas do mesmo Cliente cairiam no
  ajuste e na medição interna, que é o vazamento que a seção 3 existe para evitar.
- `class_weight=None`, ao contrário da logística da seção 2.3. Lá a reponderação era necessária
  para o piso não colapsar na classe majoritária. Aqui as duas métricas usadas dependem apenas da
  ordenação do score, que a reponderação não melhora, e ela deslocaria a probabilidade predita
  para longe da frequência observada, o que estragaria o escore de Brier da seção 6 e o limiar
  por capacidade da seção 7.

A semente é a mesma `SEMENTE` da seção 1.1, e é ela que faz duas execuções seguidas devolverem o
mesmo número.


In [ ]:
import modelo
importlib.reload(modelo)
from modelo import (HIPERPARAMETROS_CANDIDATO, comparar_nos_folds,
                    conferir_ganho_sobre_os_pisos, criar_candidato)

for nome, valor in HIPERPARAMETROS_CANDIDATO.items():
    print(f"{nome}: {valor}")
print(f"random_state: {SEMENTE}")

# O candidato final vê a partição de treino inteira, com o pré-processador já
# ajustado na seção 1.4. É este objeto que as seções 5 a 8 consomem.
candidato = criar_candidato(semente=SEMENTE)
candidato.fit(x_treino_transformado, y_treino)

# `n_iter_` é igual a max_iter por construção, porque `early_stopping=False`
# desliga a parada antecipada. O print não mede nada que varie, serve de
# conferência da decisão: se a parada antecipada voltar a ligar sozinha, os dois
# números deixam de bater e isso aparece aqui.
print(f"\niterações de boosting: {candidato.n_iter_}"
      f" (max_iter declarado: {HIPERPARAMETROS_CANDIDATO['max_iter']})")


### 4.2. Desempenho na validação contra os dois pisos

A comparação acontece nos cinco folds da seção 3, e os três modelos passam pelos mesmos folds.
Duas decisões sustentam o número:

- **Fábrica, e não modelo pronto.** Cada fold recebe um estimador novo. Reaproveitar a instância
  já ajustada faria o segundo fold começar do ajuste do primeiro, e a média dos cinco mediria um
  modelo que já viu quase todo o treino.
- **Pré-processador reajustado dentro de cada fold.** O da seção 1.4 é ajustado na partição de
  treino inteira, o que é correto para o modelo final, mas usá-lo aqui deixaria a mediana da
  imputação e a escala carregarem informação das linhas que o fold usa como validação. O efeito
  seria pequeno e invisível, a pior combinação possível num número que decide hiperparâmetro na
  seção seguinte.

Os dois pisos são os mesmos da seção 2, com a mesma parametrização, para que a diferença medida
seja de algoritmo e não de configuração. A precisão média é a métrica da comparação porque
resume a curva de precisão contra revocação, que é a leitura adequada quando só um em cada cinco
Clientes é Detrator.

Esta é a célula mais cara do notebook: são quinze ajustes, cinco por algoritmo.


In [ ]:
from sklearn.base import clone

# As fábricas dos dois pisos clonam os objetos das células 2.2 e 2.3 em vez de
# repetir a configuração deles. `clone` devolve um estimador novo, ainda não
# ajustado, com os mesmos parâmetros, que é exatamente o que `avaliar_nos_folds`
# pede. Copiar a configuração aqui deixaria um ajuste na logística da seção 2
# fora da comparação por folds, e as duas medidas passariam a falar de modelos
# diferentes sem nada no caminho avisar.
fabricas = {
    "classe majoritária": lambda: clone(trivial),
    "regressão logística": lambda: clone(logistica),
    "gradient boosting": lambda: criar_candidato(semente=SEMENTE),
}

comparacao_validacao = comparar_nos_folds(
    fabricas, x_treino, y_treino, folds, preprocessador,
)

# CR02 do #103 como trava: sem ela, um candidato pior que o piso seguiria para o
# ajuste da seção 5 e só a tabela da seção 9 denunciaria o problema.
conferir_ganho_sobre_os_pisos(
    comparacao_validacao,
    candidato="gradient boosting",
    pisos=["classe majoritária", "regressão logística"],
)

display(comparacao_validacao)


**Leitura do resultado.** Medida na base analítica atual, a média dos cinco folds:

| Modelo | Precisão média | Desvio entre folds | ROC-AUC |
|---|---|---|---|
| classe majoritária | 0,2043 | 0,0011 | 0,5000 |
| regressão logística | 0,4766 | 0,0061 | 0,7337 |
| gradient boosting | 0,5023 | 0,0060 | 0,7424 |

O candidato supera os dois pisos, e o que sustenta a afirmação não é a média sozinha: ele fica à
frente da regressão logística **nos cinco folds, um a um**, com ganho de 0,026 na precisão média,
pouco mais de quatro vezes o desvio entre folds. Uma diferença menor que esse desvio não
autorizaria a conclusão, e é por isso que o desvio aparece na tabela.

O piso trivial se comporta como a seção 2.2 previu: precisão média de 0,2043, que reproduz a
prevalência de Detrator no treino, e ROC-AUC de 0,5, o valor de quem não ordena nada.

**O tamanho do ganho, dito com honestidade.** A distância entre o candidato e a logística é de
0,026, enquanto a distância entre a logística e o piso trivial é de 0,272. A maior parte do sinal
desta base já é linear, e a interação entre fidelização e falha operacional acrescenta uma fração
menor, consistente mas menor. Isso não desfaz a escolha do algoritmo, que segue justificada
porque o ganho é real e aparece em todos os folds, mas dimensiona o que ela entrega: a redação da
Seção 4.3 deve apresentar o boosting como um avanço medido, não como um salto. O ajuste de
hiperparâmetros da seção 5 é a próxima chance de mover esse número, e é ele que dirá se 0,026 é o
teto do método nesta base ou apenas o resultado da configuração inicial.

## 5. Ajuste de hiperparâmetros

Busca sobre taxa de aprendizado, tamanho da árvore e número de iterações, avaliada apenas nos
folds da seção 3. O conjunto de teste não participa e segue intocado até a seção 6.

O objetivo não é maximizar a métrica. É responder a pergunta que a seção 4.2 deixou aberta: os
0,026 de vantagem do candidato sobre a regressão logística eram o teto do método nesta base, ou
apenas o resultado de uma configuração inicial escolhida no papel? A resposta precisa ficar
rastreável, porque a seção 4.4 vai comparar modelos e vai precisar tirar o número daqui em vez
de reexecutar a busca.

### 5.1. Grade declarada e isolamento do conjunto de teste

`GRADE_HIPERPARAMETROS` (`src/modelo.py`) é fatorial e pequena de propósito: doze combinações
sobre três eixos. Assim cada efeito fica visível isoladamente e a tabela cabe no documento. Uma
busca aleatória maior daria talvez um número melhor, sem deixar claro qual parâmetro o produziu,
que é o oposto do que esta seção precisa entregar.

A configuração do #103 (`learning_rate` 0,05, `max_leaf_nodes` 31, `max_iter` 300) é uma das doze
células, e não um ponto medido por fora. Sem ela dentro da mesma tabela, passando pelos mesmos
folds, o ganho da busca não seria comparável: estaria sendo confrontado com um número produzido
em outra execução.

A decisão usa **precisão média**, calculada por `average_precision_score` em cada fold e depois
promediada entre os cinco folds. Essa métrica resume a curva de precisão contra revocação e é a
leitura adequada quando os Detratores são minoria; acurácia não decide a grade porque pode premiar
um modelo que nunca identifica a classe rara. O ROC-AUC é registrado como diagnóstico secundário,
mas não escolhe a configuração.

**Por que o teste não entra, e como isso se verifica.** `buscar_hiperparametros` não tem
parâmetro por onde receber a partição de teste, e os folds vêm de `validacao.criar_folds`, que
particiona apenas o treino agrupando por `ID_GOLDENRECORD`. A célula abaixo valida com `assert` a
cobertura das posições, a separação entre treino e teste e a ausência de um mesmo Cliente nos dois
lados de cada fold. A busca, duas células adiante, salva a tabela agregada em
`assets/hiperparametros_candidato.json`; é JSON e não CSV porque o projeto bloqueia arquivos CSV
no Git para proteger os dados do parceiro.

In [ ]:
from modelo import (GRADE_HIPERPARAMETROS, buscar_hiperparametros,
                    combinacoes_da_grade, escolher_configuracao)

for eixo, valores in GRADE_HIPERPARAMETROS.items():
    print(f"{eixo}: {valores}")
combinacoes = combinacoes_da_grade(GRADE_HIPERPARAMETROS)
print(f"combinações: {len(combinacoes)} | ajustes: {len(combinacoes) * len(folds)}")

# Prova do CR01. Os asserts fazem a execução quebrar se a cobertura, a separação
# entre treino e teste ou o agrupamento por Cliente deixar de valer.
posicoes = np.unique(np.concatenate([np.concatenate(fold) for fold in folds]))
linhas_em_comum = x_treino.index.intersection(x_teste.index)
clientes_nos_dois_lados = max(
    len(set(grupos_treino.iloc[ajuste]) & set(grupos_treino.iloc[validacao]))
    for ajuste, validacao in folds
)
assert np.array_equal(np.sort(posicoes), np.arange(len(x_treino)))
assert linhas_em_comum.empty
assert clientes_nos_dois_lados == 0

print(f"\nposições cobertas pelos folds: {len(posicoes):,}".replace(",", ".")
      + f" de {len(x_treino):,}".replace(",", ".") + " linhas do treino")
print("linhas em comum entre treino e teste:", len(linhas_em_comum))
print("maior número de ID_GOLDENRECORD nos dois lados de um mesmo fold:",
      clientes_nos_dois_lados)

### 5.2. Busca nos folds de validação

Cada combinação passa pelos **mesmos** cinco folds, com o pré-processador reajustado dentro de
cada um, pelas razões já detalhadas na seção 4.2. Sem isso, a diferença entre duas linhas da
tabela misturaria efeito de hiperparâmetro com efeito de partição, e a escolha final seria feita
sobre essa mistura.

A tabela sai inteira, uma linha por combinação, e não apenas o vencedor: é o que permite
reconstruir a decisão sem repetir os quinze minutos de execução.

Esta é a célula mais cara do notebook: sessenta ajustes, doze combinações por cinco folds.

In [ ]:
tabela_busca = buscar_hiperparametros(
    GRADE_HIPERPARAMETROS, x_treino, y_treino, folds, preprocessador, semente=SEMENTE,
)
tabela_busca_ordenada = tabela_busca.sort_values(
    "precisao_media", ascending=False,
).reset_index(drop=True)
# O caminho sai de `caminho_base`, resolvido na seção 1.2, para que a gravação
# funcione igual rodando da raiz do projeto, de dentro de notebooks/ ou no Colab.
PASTA_ASSETS = caminho_base.parents[2] / "assets"
PASTA_ASSETS.mkdir(exist_ok=True)

destino_busca = PASTA_ASSETS / "hiperparametros_candidato.json"
tabela_busca_ordenada.to_json(destino_busca, orient="records", indent=2)
print("tabela agregada salva em", destino_busca)
display(tabela_busca_ordenada)

**Leitura da tabela.** Medida na base analítica atual, ordenada da melhor para a pior. A tabela
abaixo arredonda para quatro casas; as doze combinações sem arredondamento ficam salvas pela
própria célula da busca em `assets/hiperparametros_candidato.json`, para que a decisão possa ser
reconstruída sem repetir os quinze minutos de execução:

| `learning_rate` | `max_leaf_nodes` | `max_iter` | Precisão média | Desvio entre folds | ROC-AUC |
|---:|---:|---:|---:|---:|---:|
| 0,05 | 31 | 150 | 0,5025 | 0,0059 | 0,7426 |
| 0,05 | 31 | 300 | 0,5023 | 0,0060 | 0,7424 |
| 0,05 | 63 | 150 | 0,5021 | 0,0056 | 0,7422 |
| 0,10 | 31 | 150 | 0,5020 | 0,0060 | 0,7424 |
| 0,05 | 31 | 600 | 0,5013 | 0,0060 | 0,7416 |
| 0,10 | 31 | 300 | 0,5010 | 0,0059 | 0,7417 |
| 0,05 | 63 | 300 | 0,5009 | 0,0054 | 0,7414 |
| 0,10 | 63 | 150 | 0,5009 | 0,0055 | 0,7413 |
| 0,10 | 31 | 600 | 0,4985 | 0,0056 | 0,7399 |
| 0,05 | 63 | 600 | 0,4984 | 0,0056 | 0,7395 |
| 0,10 | 63 | 300 | 0,4981 | 0,0055 | 0,7394 |
| 0,10 | 63 | 600 | 0,4930 | 0,0053 | 0,7358 |

O resultado é consistente nos três eixos dentro da grade principal, e menor capacidade é melhor
nas comparações diretas:

- `learning_rate` 0,05 supera 0,10 nas seis comparações em que o resto é igual.
- `max_leaf_nodes` 31 supera 63 nas seis comparações em que o resto é igual.
- `max_iter` cai de 150 para 300 e para 600 nos quatro grupos de `learning_rate` e `max_leaf_nodes`.

A configuração do #103 já estava, portanto, do lado certo dos três eixos, e ligeiramente passada
do ponto apenas em `max_iter`. Não há região da grade principal em que aumentar a capacidade do
modelo melhore a validação: o boosting já esgotou o sinal que esta matriz de features oferece e o
que vem depois é ajuste ao ruído do fold de ajuste.

**O tamanho de tudo isso.** A grade inteira, da melhor à pior célula, cobre 0,0095 de precisão
média. O desvio entre folds de uma única configuração é de cerca de 0,0059. Ou seja, **a
distância entre o melhor e o pior hiperparâmetro testado é menos de duas vezes o ruído de
medição**. Para comparação, a troca de algoritmo medida na seção 4.2, da logística para o
boosting, valeu 0,026, mais de quatro vezes esse mesmo desvio. A escolha do algoritmo mudou o
resultado cerca de duas vezes e meia mais do que a grade de hiperparâmetros inteira.

### 5.3. Verificação de contorno e configuração final

O melhor ponto da grade, `max_iter` 150, é o menor valor testado nesse eixo, e vencedor na borda
é motivo de desconfiança: pode ser que o ótimo estivesse fora da grade, num valor ainda menor que
não foi medido. A célula abaixo fecha essa dúvida antes da escolha, medindo dois valores abaixo
de 150 no eixo vencedor. É uma verificação de contorno declarada, e não uma ampliação disfarçada
da grade: ela não entra na tabela da decisão.

Em seguida, `escolher_configuracao` aplica o critério. A regra é a mesma que o #103 usou para
afirmar que o candidato supera a logística: **ganho menor que o desvio entre folds não sustenta a
afirmação de que uma configuração é melhor que a outra**. Aqui ela vira critério de escolha, e
não apenas de leitura. Se o melhor da grade não superar a configuração de partida por mais que o
desvio dela, a de partida permanece, porque trocar a configuração final do modelo para perseguir
a terceira casa decimal seria escolher ruído.

In [ ]:
# Contorno no eixo vencedor, abaixo do menor valor da grade.
contorno = buscar_hiperparametros(
    {"learning_rate": [0.05], "max_leaf_nodes": [31], "max_iter": [50, 100]},
    x_treino, y_treino, folds, preprocessador, semente=SEMENTE,
)
destino_contorno = PASTA_ASSETS / "contorno_hiperparametros.json"
contorno.to_json(destino_contorno, orient="records", indent=2)
print("tabela de contorno salva em", destino_contorno)
display(contorno)

escolha = escolher_configuracao(tabela_busca)
for chave, valor in escolha.items():
    print(f"{chave}: {valor}")

**Conclusão do ajuste.** A verificação de contorno mostra que o ótimo está cercado, e não fora da
grade: a precisão média sobe de 0,4982 em `max_iter` 50 para 0,5020 em 100 e 0,5025 em 150, e
volta a cair em 300 e 600. O eixo tem um máximo interno, e a grade o continha. A tabela de
contorno fica salva em `assets/contorno_hiperparametros.json`.

O melhor ponto medido é `learning_rate` 0,05, `max_leaf_nodes` 31 e `max_iter` 150, com precisão
média de 0,502487, contra 0,502337 da configuração do #103. **O ganho é de 0,00015, cerca de
quarenta vezes menor que o desvio de 0,0060 entre os folds daquela configuração.** Nas quatro
casas da tabela acima as duas linhas aparecem como 0,5025 e 0,5023, o que sugeriria 0,0002; o
valor sem arredondamento é o que a célula acima imprime em `ganho_sobre_a_partida` e o que fica
em `assets/hiperparametros_candidato.json`. Pelo critério declarado acima, isso não é ganho: é a
mesma medida repetida em dois pontos vizinhos de um platô.

**A configuração do #103 permanece como configuração final do candidato:** `learning_rate` 0,05,
`max_leaf_nodes` 31, `max_iter` 300, com os demais hiperparâmetros como declarados em
`HIPERPARAMETROS_CANDIDATO`. Ela fica dentro do platô que vai de `max_iter` 100 a 300, e trocá-la
por 150 mudaria o modelo documentado na seção 4 sem que nada tivesse melhorado de forma
mensurável.

**O que este card entrega para a seção 4.4, dito sem inflar.** O ajuste de hiperparâmetros não
melhorou o candidato, e esse é o resultado, não uma falha da busca. Ele responde a pergunta que a
seção 4.2 deixou aberta: os 0,026 de vantagem sobre a logística são o teto do método nesta base,
e não o efeito de uma configuração inicial mal escolhida. O caminho para mover esse número não
passa por hiperparâmetro; passa por feature, que é onde o #141 e a discussão da seção 4.3 já
apontavam. Registrar isso vale mais para a comparação de modelos do que teria valido um ganho de
terceira casa decimal apresentado como melhoria.

## 6. Métricas de ordenação do score

Precisão média, área sob a curva ROC e escore de Brier, calculadas sobre o conjunto de teste.

*A preencher no card #105.*


## 7. Limiar operacional e matriz de confusão

Ponto de corte derivado da capacidade diária de contato da equipe de Experiência do Cliente, e não fixado em 0,5 por omissão.

*A preencher no card #106.*


## 8. Curvas ROC e de precisão contra cobertura

Figuras salvas em `assets/`, no mesmo padrão das da seção 4.2.

*A preencher no card #107.*


## 9. Tabela comparativa

Baselines e modelo candidato nas mesmas métricas e sobre a mesma partição de teste.

*A preencher no card #108.*
